# Laboratorio: distancia y proyecciones

En este laboratorio combinaremos cálculo exacto, comprobación numérica y visualización. Al finalizar podrá:

1. verificar la caracterización de una proyección convexa;
2. proyectar sobre subespacios con bases ortonormales y no ortonormales;
3. comprobar las propiedades de una matriz de proyección;
4. trabajar con conjuntos afines, hiperplanos y núcleos.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sp.init_printing(use_unicode=True)

## 1. Proyección sobre un segmento convexo

Sea

$$C=\{(t,0):0\leq t\leq2\},\qquad x=(3,1).$$

El punto más cercano es $p=(2,0)$. Como $C$ es convexo pero no es un subespacio, la caracterización usa

$$\langle x-p,y-p\rangle\leq0\qquad(y\in C).$$

In [ ]:
x_convexo = np.array([3.0, 1.0])
p_convexo = np.array([2.0, 0.0])
t_valores = np.linspace(0.0, 2.0, 101)
puntos_C = np.column_stack([t_valores, np.zeros_like(t_valores)])

productos = (puntos_C - p_convexo) @ (x_convexo - p_convexo)
distancias = np.linalg.norm(puntos_C - x_convexo, axis=1)

print('Máximo de los productos internos:', productos.max())
print('Punto muestreado de menor distancia:', puntos_C[np.argmin(distancias)])
print('Distancia mínima:', distancias.min())

assert np.all(productos <= 1e-12)
assert np.allclose(puntos_C[np.argmin(distancias)], p_convexo)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(puntos_C[:, 0], puntos_C[:, 1], linewidth=5, label='$C$')
ax.scatter(*x_convexo, color='tab:red', label='$x$')
ax.scatter(*p_convexo, color='tab:blue', label='$p=P_C(x)$')
ax.plot([x_convexo[0], p_convexo[0]], [x_convexo[1], p_convexo[1]], '--', color='gray')
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend()
ax.set_title('Proyección sobre un segmento convexo')
plt.show()

## 2. Proyección exacta sobre un subespacio

Sean

$$b_1=(1,1,0)^T,\qquad b_2=(0,1,1)^T,\qquad W=\operatorname{span}\{b_1,b_2\}.$$

La base no es ortogonal. Si $B=[b_1\ b_2]$, usamos

$$P=B(B^TB)^{-1}B^T.$$

In [ ]:
B = sp.Matrix([[1, 0],
               [1, 1],
               [0, 1]])
x = sp.Matrix([2, 0, 3])

P = sp.simplify(B * (B.T * B).inv() * B.T)
p = sp.simplify(P * x)
r = sp.simplify(x - p)

display(Markdown(r'**Matriz de proyección $P$:**'))
display(P)
display(Markdown(r'**Proyección $p=Px$:**'))
display(p)
display(Markdown(r'**Residuo $r=x-p$:**'))
display(r)
display(Markdown(r'**Comprobación $B^Tr=0$:**'))
display(sp.simplify(B.T * r))

P_esperada = sp.Matrix([[sp.Rational(2,3), sp.Rational(1,3), -sp.Rational(1,3)],
                        [sp.Rational(1,3), sp.Rational(2,3),  sp.Rational(1,3)],
                        [-sp.Rational(1,3), sp.Rational(1,3), sp.Rational(2,3)]])
assert P == P_esperada
assert p == sp.Matrix([sp.Rational(1,3), sp.Rational(5,3), sp.Rational(4,3)])
assert B.T * r == sp.zeros(2, 1)

### La misma proyección después de Gram–Schmidt

Al ortonormalizar las columnas de $B$ obtenemos $Q$. La fórmula se simplifica a $P=QQ^T$.

In [ ]:
b1, b2 = B[:, 0], B[:, 1]
q1 = sp.simplify(b1 / sp.sqrt(b1.dot(b1)))
u2 = sp.simplify(b2 - (b2.dot(q1)) * q1)
q2 = sp.simplify(u2 / sp.sqrt(u2.dot(u2)))
Q = sp.Matrix.hstack(q1, q2)
P_desde_Q = sp.simplify(Q * Q.T)

display(Markdown(r'**Base ortonormal $Q$:**'))
display(Q)
display(Markdown(r'**Comprobaciones $Q^TQ=I$ y $QQ^T=P$:**'))
display(sp.simplify(Q.T * Q), P_desde_Q)

assert sp.simplify(Q.T * Q) == sp.eye(2)
assert P_desde_Q == P

## 3. Propiedades de la matriz de proyección

Comprobaremos simetría, idempotencia, acción identidad sobre $W$ y anulación del residuo ortogonal.

In [ ]:
comprobaciones = {
    'P simétrica': P.T == P,
    'P idempotente': sp.simplify(P * P) == P,
    'P fija las columnas de B': sp.simplify(P * B) == B,
    'P anula el residuo': sp.simplify(P * r) == sp.zeros(3, 1),
    'I-P proyecta sobre el complemento': sp.simplify((sp.eye(3) - P) * x) == r
}
comprobaciones

## 4. Proyección sobre un conjunto afín

Para $A=a+W$, trasladamos el problema al origen:

$$P_A(z)=a+P_W(z-a).$$

In [ ]:
a = sp.Matrix([1, -1, 2])
z = sp.Matrix([4, 2, -2])
p_afin = sp.simplify(a + P * (z - a))
r_afin = sp.simplify(z - p_afin)

display(Markdown(r'**Punto proyectado sobre $a+W$:**'))
display(p_afin)
display(Markdown(r'**Residuo y comprobación de ortogonalidad:**'))
display(r_afin, sp.simplify(B.T * r_afin))

assert B.T * r_afin == sp.zeros(2, 1)
# p_afin-a debe pertenecer a W; se comprueba resolviendo sus coordenadas.
assert B.gauss_jordan_solve(p_afin - a)[0] == (B.T * B).inv() * B.T * (z - a)

## 5. Proyección y distancia a un hiperplano

Para

$$H=\{y:\langle y,u\rangle=c\},$$

la corrección desde $x$ hacia $H$ ocurre en la dirección normal $u$.

In [ ]:
def proyectar_hiperplano(x, u, c):
    x, u = sp.Matrix(x), sp.Matrix(u)
    if u.dot(u) == 0:
        raise ValueError('El vector normal no puede ser cero.')
    alpha = sp.simplify((x.dot(u) - c) / u.dot(u))
    p = sp.simplify(x - alpha * u)
    distancia = sp.simplify(abs(x.dot(u) - c) / sp.sqrt(u.dot(u)))
    return p, distancia

u = sp.Matrix([1, -2, 2])
c = sp.Integer(3)
x_h = sp.Matrix([4, 0, -1])
p_h, d_h = proyectar_hiperplano(x_h, u, c)

display(Markdown(r'**Proyección y distancia:**'))
display(p_h, d_h)
display(Markdown(r'**Comprobaciones $\langle p,u\rangle=c$ y $x-p\parallel u$:**'))
display(sp.simplify(p_h.dot(u)), sp.simplify(x_h - p_h))

assert p_h == sp.Matrix([sp.Rational(37,9), -sp.Rational(2,9), -sp.Rational(7,9)])
assert d_h == sp.Rational(1,3)
assert p_h.dot(u) == c

In [ ]:
# Ejemplo bidimensional: H = {(y1,y2): 2y1+y2=3}.
x0 = np.array([3.0, 2.0])
normal = np.array([2.0, 1.0])
c2 = 3.0
p0 = x0 - ((x0 @ normal - c2) / (normal @ normal)) * normal

s = np.linspace(-1.0, 3.0, 100)
recta = 3.0 - 2.0 * s
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(s, recta, label='$2y_1+y_2=3$')
ax.scatter(*x0, color='tab:red', label='$x$')
ax.scatter(*p0, color='tab:blue', label='$P_H(x)$')
ax.plot([x0[0], p0[0]], [x0[1], p0[1]], '--', color='gray')
ax.set_aspect('equal')
ax.set_xlim(-1, 3.5)
ax.set_ylim(-3, 4)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title('El residuo sigue la dirección normal')
plt.show()

assert np.allclose(p0, [1.0, 1.0])
assert np.isclose(p0 @ normal, c2)

## 6. Proyección sobre el núcleo de una matriz

Si $W=\operatorname{Nul}(A)$, podemos obtener una base del núcleo, ortonormalizarla y formar $QQ^T$. Para una matriz con una sola fila no nula también podemos usar

$$P_{\operatorname{Nul}(A)}=I-\frac{A^TA}{AA^T}.$$

In [ ]:
def gram_schmidt_exacto(vectores):
    ortogonales = []
    for v in vectores:
        v = sp.Matrix(v)
        w = v
        for q in ortogonales:
            w = sp.simplify(w - (v.dot(q) / q.dot(q)) * q)
        if w == sp.zeros(v.rows, 1):
            raise ValueError('Los vectores deben ser independientes.')
        ortogonales.append(w)
    return [sp.simplify(w / sp.sqrt(w.dot(w))) for w in ortogonales]

A = sp.Matrix([[1, 2, -1]])
x_n = sp.Matrix([2, 1, 3])
normal_col = A.T
P_nucleo = sp.simplify(sp.eye(3) - normal_col * normal_col.T / (normal_col.T * normal_col)[0])
p_nucleo = sp.simplify(P_nucleo * x_n)

base_nucleo = A.nullspace()
Q_nucleo = sp.Matrix.hstack(*gram_schmidt_exacto(base_nucleo))
P_desde_base = sp.simplify(Q_nucleo * Q_nucleo.T)

display(Markdown(r'**Base del núcleo y base ortonormal:**'))
display(base_nucleo, Q_nucleo)
display(Markdown(r'**Proyección sobre el núcleo:**'))
display(p_nucleo)
display(Markdown(r'**Comprobación $Ap=0$:**'))
display(sp.simplify(A * p_nucleo))

assert P_desde_base == P_nucleo
assert A * p_nucleo == sp.zeros(1, 1)

## 7. Implementación numérica con QR

En punto flotante evitamos formar explícitamente $(B^TB)^{-1}$. Una factorización QR produce directamente una base ortonormal de las columnas de $B$.

In [ ]:
def proyectar_subespacio(B, x, tolerancia=1e-12):
    B = np.asarray(B, dtype=float)
    x = np.asarray(x, dtype=float)
    if B.ndim != 2 or x.shape != (B.shape[0],):
        raise ValueError('Las dimensiones de B y x no son compatibles.')
    if np.linalg.matrix_rank(B, tol=tolerancia) != B.shape[1]:
        raise ValueError('Las columnas de B deben ser independientes.')
    Q, _ = np.linalg.qr(B, mode='reduced')
    P = Q @ Q.T
    return P @ x, P

p_num, P_num = proyectar_subespacio(np.array(B, dtype=float), np.array(x, dtype=float).reshape(-1))
print('Proyección numérica:', p_num)
print('Error de simetría:', np.linalg.norm(P_num.T - P_num))
print('Error de idempotencia:', np.linalg.norm(P_num @ P_num - P_num))

assert np.allclose(p_num, np.array(p, dtype=float).reshape(-1))
assert np.linalg.norm(P_num.T - P_num) < 1e-12
assert np.linalg.norm(P_num @ P_num - P_num) < 1e-12

## 8. Ejercicios

1. Cambie el punto $x$ del segmento convexo y determine cuándo la proyección cae en un extremo y cuándo cae en el interior.
2. Reemplace la base de $W$ por otra base del mismo subespacio y compruebe que la matriz $P$ no cambia.
3. Calcule $P_{W^\perp}$ como $I-P_W$ y verifique la descomposición ortogonal de tres vectores distintos.
4. Proyecte un punto sobre el hiperplano $y_1+y_2+y_3=1$ y verifique la fórmula de distancia.
5. Elija una matriz $A$ de tamaño $2\times4$, obtenga una base de $\operatorname{Nul}(A)$ y adapte el procedimiento de la sección 6.